## imports:

In [1]:
import copy
import datetime as dt
import json
import re
import time
from types import SimpleNamespace


### GIVEN:

In [2]:
# GIVEN — do not edit this cell (the grader uses its own copy)
import json, re, copy, time, datetime as dt
from types import SimpleNamespace

# The salon's clock is INJECTED. Never call datetime.now() in your assistant: use the `clock` you are given.
NOW = dt.datetime(2026, 10, 19, 10, 0)            # Monday 19 October 2026, 10:00

def fmt(t):      return t.strftime('%Y-%m-%d %H:%M')
def parse_dt(s): return dt.datetime.strptime(s, '%Y-%m-%d %H:%M')

SERVICES = {   # minutes are a multiple of 30; prices are INTEGER CENTS
    'cut':        {'name': 'Haircut',      'minutes': 30,  'price_cents': 3500},
    'beard':      {'name': 'Beard trim',   'minutes': 30,  'price_cents': 2000},
    'wash_style': {'name': 'Wash & style', 'minutes': 60,  'price_cents': 4500},
    'color':      {'name': 'Colour',       'minutes': 120, 'price_cents': 9000},
    'highlights': {'name': 'Highlights',   'minutes': 150, 'price_cents': 13000},
}
STYLISTS = {   # weekday(): Monday=0 ... Sunday=6
    'mara': {'name': 'Mara', 'services': {'cut', 'wash_style', 'color', 'highlights'}, 'days_off': {6}},
    'jon':  {'name': 'Jon',  'services': {'cut', 'beard', 'wash_style'},               'days_off': {2, 6}},
    'lea':  {'name': 'Lea',  'services': {'cut', 'color', 'highlights'},               'days_off': {0, 6}},
}
OPEN_H, CLOSE_H, SLOT_MIN, MAX_ACTIVE = 9, 18, 30, 3
SEED = [   # (customer, service, stylist, start)
    ('c_ada', 'cut',        'mara', '2026-10-20 10:00'),   # B-1  exactly 24h from NOW
    ('c_ada', 'color',      'lea',  '2026-10-23 14:00'),   # B-2
    ('c_ben', 'beard',      'jon',  '2026-10-19 15:00'),   # B-3  today, 5h away
    ('c_cy',  'cut',        'jon',  '2026-10-20 13:00'),   # B-4
    ('c_cy',  'wash_style', 'mara', '2026-10-22 11:00'),   # B-5
    ('c_cy',  'beard',      'jon',  '2026-10-24 16:00'),   # B-6  (c_cy has the maximum of 3 active bookings)
]

class BackendTimeout(Exception):
    """The salon system did not answer in time (the request may or may not have been applied for writes)."""
class SlotTaken(Exception):
    """Somebody else holds that slot."""
class BackendError(Exception):
    """An expected business refusal; .code is a short machine-readable reason."""
    def __init__(self, code):
        super().__init__(code); self.code = code

class SalonBackend:
    """The salon's booking system. READ it, do not edit it. Every public method is a possible failure point."""
    def __init__(self, now=NOW):
        self.now = now
        self.bookings, self.by_key, self.tickets = {}, {}, []
        self.next_id = 1
        self.call_log, self.write_log = [], []          # every method called / every COMMITTED write
        self.faults = {'fail_reads': 0, 'fail_next_write': 0, 'lose_next_write_response': 0}
        for customer, service, stylist, start in SEED:
            self._insert(customer, service, stylist, start)

    # ---------- helpers ----------
    def _log(self, name): self.call_log.append(name)
    def _read_fault(self):
        if self.faults['fail_reads'] > 0:
            self.faults['fail_reads'] -= 1
            raise BackendTimeout('read timeout')
    def _insert(self, customer, service, stylist, start):
        bid = f'B-{self.next_id}'; self.next_id += 1
        self.bookings[bid] = {'booking_id': bid, 'customer_id': customer, 'service_id': service, 'stylist_id': stylist,
                              'start': start, 'status': 'active', 'reschedules': 0, 'fee_cents': 0}
        return self.bookings[bid]
    def _busy(self, stylist, ignore=None):
        out = []
        for b in self.bookings.values():
            if b['status'] == 'active' and b['stylist_id'] == stylist and b['booking_id'] != ignore:
                s = parse_dt(b['start']); out.append((s, s + dt.timedelta(minutes=SERVICES[b['service_id']]['minutes'])))
        return out
    def _is_free(self, service, stylist, start, ignore=None):
        s = parse_dt(start); e = s + dt.timedelta(minutes=SERVICES[service]['minutes'])
        if s <= self.now or s.weekday() in STYLISTS[stylist]['days_off'] or service not in STYLISTS[stylist]['services']:
            return False
        if s.hour < OPEN_H or e > s.replace(hour=CLOSE_H, minute=0) or s.minute % SLOT_MIN:
            return False
        return all(e <= bs or s >= be for bs, be in self._busy(stylist, ignore))
    def _fee(self, booking):
        price = SERVICES[booking['service_id']]['price_cents']
        delta = parse_dt(booking['start']) - self.now
        if delta <= dt.timedelta(0): raise BackendError('already_started')
        if delta >= dt.timedelta(hours=24): return 0
        if delta >= dt.timedelta(hours=2): return price // 2
        return price
    def _replay(self, key, sig):
        entry = self.by_key.get(key)
        if entry is None: return None
        if entry['sig'] != sig: raise BackendError('key_reuse')
        return copy.deepcopy(entry['result'])
    def _write_faults(self):
        if self.faults['fail_next_write'] > 0:
            self.faults['fail_next_write'] -= 1
            raise BackendTimeout('write timeout (nothing was applied)')
    def _lost_response(self):
        if self.faults['lose_next_write_response'] > 0:
            self.faults['lose_next_write_response'] -= 1
            raise BackendTimeout('response lost (the write WAS applied)')

    # ---------- reads ----------
    def list_services(self):
        self._log('list_services'); self._read_fault(); return copy.deepcopy(SERVICES)
    def free_slots(self, service_id, date, stylist_id=None):
        self._log('free_slots'); self._read_fault()
        if service_id not in SERVICES: raise ValueError('unknown service')
        day = dt.datetime.strptime(date, '%Y-%m-%d')
        out = []
        for stylist in sorted(STYLISTS if stylist_id is None else [stylist_id]):
            t = day.replace(hour=OPEN_H)
            while t.hour < CLOSE_H:
                if self._is_free(service_id, stylist, fmt(t)):
                    out.append({'stylist_id': stylist, 'start': fmt(t)})
                t += dt.timedelta(minutes=SLOT_MIN)
        return sorted(out, key=lambda r: (r['start'], r['stylist_id']))
    def bookings_for(self, customer_id):
        self._log('bookings_for'); self._read_fault()
        return copy.deepcopy([b for b in self.bookings.values() if b['customer_id'] == customer_id])
    def find_by_key(self, request_key):
        """Look up the stored result of an earlier write. Never faults."""
        self._log('find_by_key')
        entry = self.by_key.get(request_key)
        return copy.deepcopy(entry['result']) if entry else None

    # ---------- writes (idempotent on request_key) ----------
    def book(self, customer_id, service_id, stylist_id, start, request_key):
        self._log('book'); self._write_faults()
        sig = ('book', customer_id, service_id, stylist_id, start)
        done = self._replay(request_key, sig)
        if done is None:
            if service_id not in SERVICES or stylist_id not in STYLISTS: raise BackendError('unknown_item')
            if sum(1 for b in self.bookings.values() if b['customer_id'] == customer_id and b['status'] == 'active') >= MAX_ACTIVE:
                raise BackendError('limit_reached')
            if not self._is_free(service_id, stylist_id, start): raise SlotTaken(start)
            done = copy.deepcopy(self._insert(customer_id, service_id, stylist_id, start))
            self.by_key[request_key] = {'sig': sig, 'result': done}
            self.write_log.append(('book', done['booking_id']))
        self._lost_response()
        return done
    def cancel(self, customer_id, booking_id, fee_cents, request_key):
        self._log('cancel'); self._write_faults()
        sig = ('cancel', customer_id, booking_id, fee_cents)
        done = self._replay(request_key, sig)
        if done is None:
            b = self.bookings.get(booking_id)
            if b is None or b['customer_id'] != customer_id: raise BackendError('not_found')
            if b['status'] != 'active': raise BackendError('already_cancelled')
            if fee_cents != self._fee(b): raise BackendError('fee_mismatch')
            b['status'], b['fee_cents'] = 'cancelled', fee_cents
            done = {'booking_id': booking_id, 'fee_cents': fee_cents, 'status': 'cancelled'}
            self.by_key[request_key] = {'sig': sig, 'result': done}
            self.write_log.append(('cancel', booking_id))
        self._lost_response()
        return copy.deepcopy(done)
    def reschedule(self, customer_id, booking_id, new_start, request_key):
        self._log('reschedule'); self._write_faults()
        sig = ('reschedule', customer_id, booking_id, new_start)
        done = self._replay(request_key, sig)
        if done is None:
            b = self.bookings.get(booking_id)
            if b is None or b['customer_id'] != customer_id: raise BackendError('not_found')
            if b['status'] != 'active': raise BackendError('already_cancelled')
            if parse_dt(b['start']) - self.now < dt.timedelta(hours=24): raise BackendError('too_late')
            if b['reschedules'] >= 1: raise BackendError('already_rescheduled')
            if not self._is_free(b['service_id'], b['stylist_id'], new_start, ignore=booking_id): raise SlotTaken(new_start)
            b['start'], b['reschedules'] = new_start, b['reschedules'] + 1
            done = copy.deepcopy(b)
            self.by_key[request_key] = {'sig': sig, 'result': done}
            self.write_log.append(('reschedule', booking_id))
        self._lost_response()
        return done
    def create_ticket(self, ticket):
        """Hand a case to a human. `ticket` must be a dict with customer_id, reason, summary."""
        self._log('create_ticket')
        ticket = {**ticket, 'ticket_id': f'T-{len(self.tickets) + 1}'}
        self.tickets.append(ticket)
        return ticket['ticket_id']

# every seed booking must be valid: build one backend and check the invariants once
_b = SalonBackend()
assert all(_b._is_free(b['service_id'], b['stylist_id'], b['start'], ignore=b['booking_id']) for b in _b.bookings.values()), 'seed data must be valid'
print('Salon backend ready:', len(_b.bookings), 'seed bookings. NOW =', fmt(NOW), '(Monday)')

Salon backend ready: 6 seed bookings. NOW = 2026-10-19 10:00 (Monday)


In [3]:
# GIVEN — do not edit this cell (the grader uses its own copy)
SAFE_FALLBACK = "I can't confirm that from the salon system. Let me check again, or I can connect you with a team member."

def call(cid, name, **arguments):      return {'id': cid, 'name': name, 'arguments': arguments}
def calls(*cs):                        return {'tool_calls': list(cs)}
def final(text):                       return {'final': text}

def check_transcript(messages):
    """Rules every model call must satisfy. Returns a list of violations (empty = a valid conversation)."""
    bad, open_ids, seen_ids = [], None, set()
    if not messages or messages[0].get('role') != 'system':
        bad.append('the first message must be the system prompt')
    for i, m in enumerate(messages):
        role = m.get('role')
        if role not in ('system', 'user', 'assistant', 'tool'):
            bad.append(f'message {i}: unknown role {role!r}'); continue
        if role == 'tool':
            if open_ids is None or m.get('tool_call_id') not in open_ids:
                bad.append(f'message {i}: tool message {m.get("tool_call_id")!r} does not answer a pending tool call')
            else:
                open_ids.discard(m['tool_call_id'])
            try: json.loads(m.get('content'))
            except Exception: bad.append(f'message {i}: tool content must be a JSON string')
            continue
        if open_ids:
            bad.append(f'message {i}: tool calls {sorted(open_ids)} were never answered')
        open_ids = None
        if role == 'assistant' and m.get('tool_calls'):
            ids = [c.get('id') for c in m['tool_calls']]
            if len(set(ids)) != len(ids) or seen_ids & set(ids):
                bad.append(f'message {i}: duplicate tool call id')
            seen_ids |= set(ids); open_ids = set(ids)
    if open_ids:
        bad.append(f'tool calls {sorted(open_ids)} were never answered')
    if len(messages) > 24:
        bad.append(f'{len(messages)} messages sent to the model (limit 24)')
    return bad

class TurnScriptedModel:
    """A scripted 'model'. Each scenario turn loads the steps the model will take; every model call consumes one step."""
    def __init__(self):
        self.queue, self.violations, self.max_len, self.n_calls = [], [], 0, 0
        self.tools_seen, self.snapshots, self.turn_label = None, [], ''
    def load(self, steps, label=''):
        self.queue, self.turn_label = list(steps), label
    def __call__(self, messages, tools=None):
        self.n_calls += 1
        self.tools_seen = tools
        self.max_len = max(self.max_len, len(messages))
        self.snapshots.append(copy.deepcopy(messages))
        for v in check_transcript(messages):
            self.violations.append(f'[{self.turn_label}] {v}')
        if not self.queue:
            self.violations.append(f'[{self.turn_label}] the assistant called the model more often than this scenario allows')
            return {'final': 'SCRIPT-EXHAUSTED'}
        step = self.queue.pop(0)
        return copy.deepcopy(step(messages) if callable(step) else step)
    def observations(self):
        """Every tool observation the model has been shown so far (parsed, in order of first appearance)."""
        seen, out = set(), []
        for snap in self.snapshots:
            for m in snap:
                if m.get('role') == 'tool' and m['tool_call_id'] not in seen:
                    seen.add(m['tool_call_id'])
                    try: out.append({**json.loads(m['content']), '_id': m['tool_call_id']})
                    except Exception: pass
        return out

def T(user, *steps, session='s1', before=None, after=None, has=(), lacks=(), equals=None):
    return dict(user=user, model=list(steps), session=session, before=before, after=after,
                has=list(has), lacks=list(lacks), equals=equals)

def active(backend, customer=None):
    return [b for b in backend.bookings.values() if b['status'] == 'active' and (customer is None or b['customer_id'] == customer)]
def new_bookings(backend):
    return [b for b in backend.bookings.values() if int(b['booking_id'][2:]) > len(SEED)]
def tool_events(assistant):
    return [e for e in assistant.audit if e.get('tool') != '(user)']

def run_scenario(sc, assistant_cls, trace=False):
    """Run one scenario against a fresh salon. Returns None (pass) or a short failure message."""
    backend = SalonBackend()
    if sc.get('setup'): sc['setup'](backend)
    model = TurnScriptedModel()
    try:
        assistant = assistant_cls(model, backend, lambda: backend.now)
        sessions = sc.get('sessions') or {'s1': sc.get('customer', 'c_ada')}
        for sid, cust in sessions.items():
            assistant.open_session(sid, cust)
        ctx = SimpleNamespace(backend=backend, assistant=assistant, model=model, replies=[], sessions=sessions)
        for n, turn in enumerate(sc.get('turns', []), 1):
            label = f'turn {n} ({turn["user"][:28]!r})'
            if turn['before']: turn['before'](backend)
            model.load(turn['model'], label)
            reply = assistant.handle(turn['session'], turn['user'])
            ctx.replies.append(reply)
            if trace:
                print(f'\n[{label}]\n  USER : {turn["user"]}\n  REPLY: {reply}')
            if not isinstance(reply, str): return f'{label}: handle() must return a string'
            if model.queue and not sc.get('loose'): return f'{label}: the assistant called the model fewer times than expected ({len(model.queue)} script steps unused)'
            for needle in turn['has']:
                if needle.lower() not in reply.lower(): return f'{label}: reply should contain {needle!r}; got {reply[:90]!r}'
            for needle in turn['lacks']:
                if needle.lower() in reply.lower(): return f'{label}: reply must not contain {needle!r}'
            if turn['equals'] is not None and reply != turn['equals']: return f'{label}: expected exactly {turn["equals"][:50]!r}; got {reply[:70]!r}'
            if turn['after']:
                try: turn['after'](ctx)
                except AssertionError as exc: return f'{label}: {exc}'
        if sc.get('check'):
            sc['check'](ctx)
        if model.violations: return model.violations[0]
        return None
    except AssertionError as exc:
        return f'{exc}' or 'assertion failed'
    except NotImplementedError as exc:
        return f'not implemented yet ({exc})'
    except Exception as exc:
        return f'{type(exc).__name__}: {str(exc)[:110]}'

def run_suite(scenarios, assistant_cls, title='Public suite', total_points=None, verbose=True):
    results, earned, wsum = [], 0.0, sum(s['weight'] for s in scenarios)
    for sc in scenarios:
        err = run_scenario(sc, assistant_cls)
        results.append((sc, err))
        pts = (total_points or wsum) * sc['weight'] / wsum
        earned += 0 if err else pts
        if verbose:
            print(('✓' if err is None else '✗'), f"{sc['id']} {sc['title']}", '' if err is None else f'→ {err}')
    passed = sum(1 for _, e in results if e is None)
    print(f'{title}: {passed}/{len(scenarios)} scenarios passed' + (f'  →  {earned:.1f}/{total_points} points' if total_points else ''))
    return earned

def trace(scenario_id, assistant_cls, scenarios=None):
    """Debug helper: run ONE scenario and print every user message and reply, the failure (if any), and the audit trail."""
    pool = scenarios if scenarios is not None else PUBLIC
    sc = next(s for s in pool if s['id'] == scenario_id)
    print(f"=== {sc['id']} {sc['title']} ===")
    err = run_scenario(sc, assistant_cls, trace=True)
    print('\nRESULT:', 'PASS' if err is None else f'FAIL → {err}')

In [4]:
# GIVEN — do not edit this cell (the grader uses its own copy)
def obs_by_id(ctx):
    return {o['_id']: o for o in ctx.model.observations()}

def _nothing_written(ctx):
    assert ctx.backend.write_log == [], 'a proposal must not change anything'
    assert new_bookings(ctx.backend) == []

def _propose_cut(cid='b1'):
    return T('Please book me a haircut with Mara tomorrow at 11:00',
             calls(call(cid, 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:00')),
             final('Here is what I will book: Haircut with Mara on 2026-10-20 at 11:00 for €35.00. Shall I go ahead?'),
             has=['Shall I'], after=_nothing_written)

def sc_greeting(ctx):
    assert ctx.backend.call_log == [], f'a greeting needs no backend call, but you called {ctx.backend.call_log}'
    assert tool_events(ctx.assistant) == [] and ctx.model.n_calls == 1

def sc_browse(ctx):
    o = obs_by_id(ctx)
    assert o['a1']['status'] == 'ok' and o['a2']['status'] == 'ok', o
    assert [m['tool_call_id'] for m in ctx.model.snapshots[-1] if m['role'] == 'tool'] == ['a1', 'a2']
    assert ctx.backend.write_log == []

def sc_book_happy(ctx):
    mine = new_bookings(ctx.backend)
    assert len(mine) == 1 and ctx.backend.write_log == [('book', 'B-7')], ctx.backend.write_log
    b = mine[0]
    assert (b['customer_id'], b['service_id'], b['stylist_id'], b['start']) == ('c_ada', 'cut', 'mara', '2026-10-20 11:00')
    assert 'Now: 2026-10-19 10:00' in ctx.model.snapshots[0][0]['content'], 'the system prompt must contain the line "Now: YYYY-MM-DD HH:MM" from the clock'

def sc_unclear_and_bare_yes(ctx):
    assert ctx.backend.write_log == [] and new_bookings(ctx.backend) == []

def sc_race(ctx):
    assert ctx.backend.write_log == [('book', 'B-7')], 'only the rival booking may exist'
    last = [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment' and o.get('status') == 'error'][-1]
    assert last['error_type'] == 'slot_taken', last
    alts = last['data']['alternatives']
    assert alts[:2] == [{'stylist_id': 'jon', 'start': '2026-10-20 11:00'}, {'stylist_id': 'lea', 'start': '2026-10-20 11:00'}], alts
    assert len(alts) == 3, 'offer up to 3 alternatives, nearest in time first'

def sc_cancel_free(ctx):
    assert ctx.backend.bookings['B-2']['status'] == 'cancelled' and ctx.backend.bookings['B-2']['fee_cents'] == 0
    assert ctx.backend.write_log == [('cancel', 'B-2')]

def sc_cancel_fee(ctx):
    b = ctx.backend.bookings['B-3']
    assert b['status'] == 'cancelled' and b['fee_cents'] == 1000, b

def sc_reschedule(ctx):
    b = ctx.backend.bookings['B-1']
    assert b['start'] == '2026-10-20 14:00' and b['reschedules'] == 1 and ctx.backend.write_log == [('reschedule', 'B-1')], (b, ctx.backend.write_log)
    last = [o for o in ctx.model.observations() if o.get('tool') == 'reschedule_appointment'][-1]
    assert last['status'] == 'blocked' and last['error_type'] == 'already_rescheduled', last

def sc_too_late(ctx):
    last = [o for o in ctx.model.observations() if o.get('tool') == 'reschedule_appointment'][-1]
    assert last['status'] == 'blocked' and last['error_type'] == 'too_late', last
    assert ctx.backend.write_log == []

def sc_identity(ctx):
    assert ctx.backend.bookings['B-3']['status'] == 'active'
    o = obs_by_id(ctx)
    assert o['i1']['status'] == 'not_found' and o['i2']['error_type'] == 'validation_error' and o['i3']['status'] == 'ok'
    assert 'B-3' not in json.dumps(o['i3']), "another customer's booking must never be shown"

def sc_limit(ctx):
    last = [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment'][-1]
    assert last['status'] == 'blocked' and last['error_type'] == 'limit_reached', last
    assert ctx.backend.write_log == [] and 'book' not in ctx.backend.call_log

def sc_lost_response(ctx):
    assert len(new_bookings(ctx.backend)) == 1 and ctx.backend.write_log == [('book', 'B-7')], 'exactly one booking, even though the reply was lost'
    last = [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment'][-1]
    assert last['status'] == 'ok' and last['data']['booking_id'] == 'B-7', last

def sc_read_retry(ctx):
    ev = [e for e in tool_events(ctx.assistant) if e['tool'] == 'find_slots']
    assert len(ev) == 1 and ev[0]['status'] == 'ok' and ev[0]['attempts'] == 2, ev

def sc_read_exhausted(ctx):
    assert len(ctx.backend.tickets) == 1 and ctx.backend.tickets[0]['reason'] == 'tool_failures', ctx.backend.tickets
    assert all(o['error_type'] == 'timeout' for o in ctx.model.observations()), ctx.model.observations()

def sc_grounding(ctx):
    assert ctx.replies[0] == SAFE_FALLBACK and ctx.replies[1] == SAFE_FALLBACK, ctx.replies
    assert ctx.replies[2].startswith('Yes: B-1'), ctx.replies[2]

def sc_bad_calls(ctx):
    o = obs_by_id(ctx)
    assert [(o[i]['status'], o[i].get('error_type')) for i in ('x1', 'x2', 'x3', 'x4')] == \
        [('error', 'validation_error'), ('error', 'unknown_tool'), ('error', 'validation_error'), ('error', 'validation_error')], o
    assert ctx.backend.write_log == [] and 'book' not in ctx.backend.call_log

def sc_long_chat(ctx):
    assert len(new_bookings(ctx.backend)) == 1, 'the booking made after 30 chatty turns must still work'
    assert ctx.model.max_len <= 24, f'{ctx.model.max_len} messages sent to the model'

def sc_multi_intent(ctx):
    assert ctx.backend.bookings['B-2']['status'] == 'cancelled' and new_bookings(ctx.backend) == []
    o = obs_by_id(ctx)
    assert o['m1']['status'] == 'needs_confirmation' and o['m2']['status'] == 'blocked' and o['m2']['error_type'] == 'pending_exists', o

def sc_handoff(ctx):
    assert len(ctx.backend.tickets) == 1
    t = ctx.backend.tickets[0]
    assert t['reason'] == 'customer_request' and t['customer_id'] == 'c_ada' and isinstance(t['summary'], str) and len(t['summary']) <= 300, t
    blob = json.dumps(ctx.backend.tickets) + json.dumps(ctx.assistant.audit)
    assert '@' not in blob and '1234567' not in blob, 'tickets and audit logs must not contain the e-mail address or phone number'

def sc_schemas(ctx):
    tools = ctx.model.tools_seen
    assert isinstance(tools, list) and len(tools) == 7, 'pass your 7 tool schemas as the second argument of model(...)'
    names = {t['function']['name'] for t in tools}
    assert names == {'list_services', 'find_slots', 'get_my_bookings', 'book_appointment', 'reschedule_appointment', 'cancel_appointment', 'handoff_to_human'}, names
    for t in tools:
        p = t['function']['parameters']
        assert t['type'] == 'function' and p['type'] == 'object' and p['additionalProperties'] is False, t['function']['name']
        assert not ({'customer_id', 'request_key', 'approved', 'price', 'fee_cents'} & set(p['properties'])), f"{t['function']['name']}: the model must not control identity, keys, prices or approvals"
        assert set(p.get('required', [])) <= set(p['properties'])
    assert set(next(t for t in tools if t['function']['name'] == 'book_appointment')['function']['parameters']['required']) == {'service_id', 'stylist_id', 'start'}

def sc_fee_table(ctx):
    cases = [(3500, '2026-10-20 10:00', 0), (3500, '2026-10-20 09:59', 1750), (3500, '2026-10-19 12:00', 1750), (3500, '2026-10-19 11:59', 3500),
             (2000, '2026-10-19 15:00', 1000), (9000, '2026-10-23 14:00', 0), (4500, '2026-10-19 11:30', 4500), (2001, '2026-10-19 20:00', 1000)]
    for price, start, want in cases:
        got = cancellation_fee_cents(price, parse_dt(start), NOW)
        assert got == want, f'fee({price}, {start}) = {got}, expected {want}'
    try:
        cancellation_fee_cents(3500, parse_dt('2026-10-19 10:00'), NOW)
    except ValueError:
        return
    raise AssertionError('a booking that has already started cannot be cancelled: raise ValueError')

REPLY_CASES = [('yes','yes'),('Yes, please!','yes'),('YES','yes'),('ok','yes'),('okay thanks','yes'),('sure','yes'),('y','yes'),('go ahead','yes'),
               ('that works','yes'),('sounds good','yes'),('perfect','yes'),('yes!!!','yes'),('Yes please do','yes'),('absolutely','yes'),
               ('no','no'),('No thanks','no'),('nope','no'),('nah','no'),("don't book it",'no'),('not yet','no'),('never mind','no'),('stop','no'),
               ('yes but 3pm instead','unclear'),('yes and also add a beard trim','unclear'),('maybe','unclear'),('?','unclear'),('what does it cost?','unclear'),
               ('no, yes, no','unclear'),('yes 15:00','unclear'),('cancel my friday appointment','unclear'),('I would like that but later','unclear'),
               ('','unclear'),('   ','unclear'),('ok but wait','unclear'),('yes, and can you also move B-2?','unclear'),('Can you do 4pm?','unclear')]

def sc_reply_table(ctx):
    wrong = [(t, want, interpret_reply(t)) for t, want in REPLY_CASES if interpret_reply(t) != want]
    assert not wrong, f'{len(wrong)} wrong, e.g. interpret_reply({wrong[0][0]!r}) = {wrong[0][2]!r}, expected {wrong[0][1]!r}'

def _fillers(n): return [T('ok thanks', final("You're welcome!")) for _ in range(n)]

PUBLIC = [
 dict(id='P01', title='Greeting needs no tools', weight=1, turns=[T('Hi!', final('Hello! How can I help you today?'), has=['Hello'])], check=sc_greeting),
 dict(id='P02', title='Browse: two independent reads in one turn', weight=2, turns=[T("What do you offer, and what's free for a haircut on Tuesday?",
      calls(call('a1', 'list_services'), call('a2', 'find_slots', service_id='cut', date='2026-10-20')),
      final('We offer Haircut (€35.00), Beard trim, Wash & style, Colour and Highlights. Tuesday haircut slots start at 09:00 and 09:30.'), has=['09:00', '€35'])], check=sc_browse),
 dict(id='P03', title='Book: propose, then confirm in a LATER turn', weight=3, turns=[_propose_cut(),
      T('yes please', final('Done! Your haircut with Mara is booked for 2026-10-20 at 11:00 (B-7).'), has=['B-7'])],
      check=sc_book_happy),
 dict(id='P04', title='"yes, but..." and a bare "yes" never book', weight=3, turns=[_propose_cut(),
      T('yes but make it 15:00 instead', final('Sure, let me look at 15:00 instead.')),
      T('yes', final('Which appointment do you mean?'))], check=sc_unclear_and_bare_yes),
 dict(id='P05', title='interpret_reply: 36 customer replies', weight=3, check=sc_reply_table),
 dict(id='P06', title='Race: the slot is taken between proposal and "yes"', weight=3, turns=[_propose_cut(),
      T('yes', final("I'm sorry, 11:00 was just taken. I can offer 11:00 with Jon or Lea, or 10:30 with Jon."), has=['11:00'],
        before=lambda b: b.book('c_ben', 'cut', 'mara', '2026-10-20 11:00', 'rival-1'))], check=sc_race),
 dict(id='P07', title='Cancel more than 24h ahead is free', weight=2, turns=[
      T('I need to cancel my Friday colour appointment', calls(call('c1', 'get_my_bookings')),
        calls(call('c2', 'cancel_appointment', booking_id='B-2')),
        final('Cancelling B-2 (Colour on 2026-10-23 at 14:00) is free of charge. Shall I cancel it?')),
      T('yes', final('B-2 is cancelled. No fee was charged.'), has=['cancelled'])], check=sc_cancel_free),
 dict(id='P08', title='Late cancellation: fee shown BEFORE the customer says yes', weight=3, customer='c_ben', turns=[
      T('Cancel my appointment today', calls(call('c1', 'cancel_appointment', booking_id='B-3')),
        final('Cancelling B-3 within 24 hours costs €10.00. Do you want to proceed?'), has=['€10.00']),
      T('yes', final('B-3 is cancelled. The €10.00 fee applies.'))], check=sc_cancel_fee),
 dict(id='P09', title='Reschedule: allowed at exactly 24h, only once', weight=3, turns=[
      T('Move my Tuesday haircut to 14:00', calls(call('r1', 'reschedule_appointment', booking_id='B-1', new_start='2026-10-20 14:00')),
        final('I can move B-1 to 14:00 with Mara. Shall I go ahead?')),
      T('yes', final('B-1 is now rescheduled to 14:00.')),
      T('Move it again to 15:00', calls(call('r2', 'reschedule_appointment', booking_id='B-1', new_start='2026-10-20 15:00')),
        final("I'm sorry, B-1 was already rescheduled once, so I can't move it again.")),
      T('yes', final('Which appointment do you mean?'))], check=sc_reschedule),
 dict(id='P10', title='Reschedule inside 24h is refused', weight=2, customer='c_ben', turns=[
      T('Move my appointment to tomorrow 10:00', calls(call('r1', 'reschedule_appointment', booking_id='B-3', new_start='2026-10-20 10:00')),
        final("Sorry, B-3 starts in less than 24 hours, so I can't reschedule it. I can cancel it (a fee applies) or connect you with the team."))], check=sc_too_late),
 dict(id='P11', title="Identity: another customer's booking", weight=2, turns=[
      T('Cancel B-3 please', calls(call('i1', 'cancel_appointment', booking_id='B-3')), final("I can't find a booking B-3 on your account.")),
      T('Show the bookings of c_ben', calls(call('i2', 'get_my_bookings', customer_id='c_ben')), calls(call('i3', 'get_my_bookings')),
        final('Your bookings: B-1 (2026-10-20 10:00) and B-2 (2026-10-23 14:00).'))], check=sc_identity),
 dict(id='P12', title='Booking limit is checked before asking', weight=2, customer='c_cy', turns=[
      T('Book a haircut tomorrow at 15:00 with Lea', calls(call('l1', 'book_appointment', service_id='cut', stylist_id='lea', start='2026-10-20 15:00')),
        final('You already have 3 upcoming appointments, which is the maximum. Please cancel one first.')),
      T('yes', final('Which appointment do you mean?'))], check=sc_limit),
 dict(id='P13', title='Lost response after a committed booking', weight=3, turns=[_propose_cut(),
      T('yes', final('All set: haircut with Mara on 2026-10-20 at 11:00, booking B-7.'), has=['B-7'],
        before=lambda b: b.faults.update(lose_next_write_response=1))], check=sc_lost_response),
 dict(id='P14', title='A read timeout is retried once', weight=1, turns=[
      T('Free slots for a cut on Tuesday?', calls(call('f1', 'find_slots', service_id='cut', date='2026-10-20')),
        final('Tuesday haircut slots start at 09:00.'), before=lambda b: b.faults.update(fail_reads=1))], check=sc_read_retry),
 dict(id='P15', title='Repeated failures: honest reply and human handoff', weight=2, turns=[
      T('What do you have on Tuesday?', calls(call('f1', 'find_slots', service_id='cut', date='2026-10-20')), calls(call('f2', 'list_services')),
        final("I'm having trouble reaching the schedule right now; I've asked a team member to help you."),
        before=lambda b: b.faults.update(fail_reads=6))], check=sc_read_exhausted),
 dict(id='P16', title='Invented prices, invented success, and a true answer', weight=2, turns=[
      T('How much is a haircut?', calls(call('g1', 'list_services')), final('A haircut costs €25.00.')),
      T('Do I have a booking?', final('Your appointment is booked for tomorrow!')),
      T('Do I have any booking?', calls(call('g2', 'get_my_bookings')), final('Yes: B-1 on 2026-10-20 at 10:00.'))], check=sc_grounding),
 dict(id='P17', title='Malformed, unknown and smuggling tool calls', weight=2, turns=[
      T('Book me something', calls(call('x1', 'book_appointment', service_id='cut', stylist_id='mara', start='tomorrow 11am'),
                                   call('x2', 'delete_all_bookings'),
                                   call('x3', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:00', customer_id='c_ben'),
                                   call('x4', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:15')),
        final('I need an exact time such as 2026-10-20 11:00. Which time would you like?'))], check=sc_bad_calls),
 dict(id='P18', title='30 chatty turns, then a booking (context window)', weight=2,
      turns=_fillers(30) + [_propose_cut('lb1'), T('yes', final('Booked: haircut with Mara on 2026-10-20 at 11:00 (B-7).'), has=['B-7'])], check=sc_long_chat),
 dict(id='P19', title='Two requests in one message: one pending action', weight=2, turns=[
      T('Cancel B-2 and book a cut on Saturday at 11:00 with Mara',
        calls(call('m1', 'cancel_appointment', booking_id='B-2'), call('m2', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-24 11:00')),
        final("Let's start with cancelling B-2 (free of charge). Shall I cancel it? Then we'll book Saturday.")),
      T('yes', final('B-2 is cancelled.'))], check=sc_multi_intent),
 dict(id='P20', title='Human handoff without leaking personal data', weight=2, turns=[
      T('I want a person. My email is ada@example.com and my phone is +49 170 1234567',
        calls(call('h1', 'handoff_to_human', reason='customer_request')), final("I've passed your request to our team; someone will contact you."))], check=sc_handoff),
 dict(id='P21', title='Tool schemas', weight=1, turns=[T('Hi', final('Hello!'))], check=sc_schemas),
 dict(id='P22', title='cancellation_fee_cents: boundaries', weight=2, check=sc_fee_table),
]

def run_public(assistant_cls, **kw):
    return run_suite(PUBLIC, assistant_cls, 'Public suite', total_points=40, **kw)
print(len(PUBLIC), 'public scenarios loaded.')

22 public scenarios loaded.


### cancellation fee cents:

In [5]:
def cancellation_fee_cents(price_cents, start, now):
    """Fee in cents; raise ValueError if the appointment has already started."""
    remaining = start - now
    if remaining <= dt.timedelta(0):
        raise ValueError('the appointment has already started')
    if remaining >= dt.timedelta(hours=24):
        return 0
    if remaining >= dt.timedelta(hours=2):
        return price_cents // 2
    return price_cents

### check cancellation fee cents:

In [6]:
n = dt.datetime(2026, 10, 19, 10, 0)
H = lambda h, m=0: n + dt.timedelta(hours=h, minutes=m)

assert cancellation_fee_cents(3500, H(24), n) == 0   
assert cancellation_fee_cents(3500, H(23, 59), n) == 1750
assert cancellation_fee_cents(3501, H(5), n) == 1750    
assert cancellation_fee_cents(3500, H(2), n) == 1750     
assert cancellation_fee_cents(3500, H(1, 59), n) == 3500

for bad in (n, H(-1)):
    try:
        cancellation_fee_cents(3500, bad, n)
        raise SystemExit("it should be value eror !")
    except ValueError:
        pass
print("fee OK")

fee OK


### interpert reply :

In [7]:
_YES = (
    r"(?:yes|yeah|yep|yup|y|ok|okay|sure|absolutely|definitely|perfect|great|alright|fine|"
    r"confirm|confirmed|correct|go ahead|that works(?: for me)?|(?:that )?sounds good|"
    r"do it|please do|go for it|book it)"
)
_YES_TAIL = r"(?: (?:yes|please|thanks|thank you|do it|do|go ahead|perfect|great|sure|ok|okay))*"
_NO = r"(?:no|nope|nah|stop|not yet|never mind|nevermind|don't|dont|do not|please don't|please dont)"
_NO_TAIL = r"(?: (?:thanks|thank you|please|it|that|book it|cancel it|do it|now))*"
YES_RE = re.compile(rf"^{_YES}{_YES_TAIL}$")
NO_RE  = re.compile(rf"^{_NO}{_NO_TAIL}$")

def interpret_reply(text):
    """Return 'yes', 'no' or 'unclear'. Conservative: when in doubt, 'unclear'."""
    if not isinstance(text, str) or '?' in text:
        return 'unclear'

    s = text.lower().replace('’', "'")
    s = re.sub(r"[^a-z0-9' ]+", ' ', s)
    s = ' '.join(s.split()) 

    if YES_RE.match(s):
        return 'yes'
    if NO_RE.match(s):
        return 'no'
    return 'unclear'


### tool schema:

In [8]:
_SERVICE_IDS = ['cut', 'beard', 'wash_style', 'color', 'highlights']
_STYLIST_IDS = ['mara', 'jon', 'lea']
_START_PATTERN = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}$'

def _tool(name, description, properties=None, required=None):
    return {'type': 'function', 'function': {
        'name': name, 'description': description,
        'parameters': {'type': 'object', 'properties': properties or {},
                       'required': required or [], 'additionalProperties': False}}}

_SERVICE = {'type': 'string', 'enum': _SERVICE_IDS, 'description': 'Service id'}
_STYLIST = {'type': 'string', 'enum': _STYLIST_IDS, 'description': 'Stylist id'}
_BOOKING = {'type': 'string', 'pattern': r'^B-\d+$', 'description': 'Booking id such as B-2'}
_START = {'type': 'string', 'pattern': _START_PATTERN,
          'description': "Local time 'YYYY-MM-DD HH:MM' on a 30-minute mark"}

TOOL_SCHEMAS = [
    _tool('list_services', 'List the salon services with durations and prices.'),
    _tool('find_slots', 'Find free appointment slots for a service on one day.',
          {'service_id': _SERVICE,
           'date': {'type': 'string', 'pattern': r'^\d{4}-\d{2}-\d{2}$', 'description': "Date 'YYYY-MM-DD'"},
           'stylist_id': _STYLIST}, ['service_id', 'date']),
    _tool('get_my_bookings', "List the logged-in customer's own bookings."),
    _tool('book_appointment', 'PROPOSE a booking. Nothing is booked until the customer confirms in their next message.',
          {'service_id': _SERVICE, 'stylist_id': _STYLIST, 'start': _START},
          ['service_id', 'stylist_id', 'start']),
    _tool('reschedule_appointment', "PROPOSE moving one of the customer's bookings to a new start time (confirmation needed).",
          {'booking_id': _BOOKING, 'new_start': _START}, ['booking_id', 'new_start']),
    _tool('cancel_appointment', "PROPOSE cancelling one of the customer's bookings (confirmation needed; a fee may apply).",
          {'booking_id': _BOOKING}, ['booking_id']),
    _tool('handoff_to_human', 'Pass the case to a human team member.',
          {'reason': {'type': 'string', 'enum': ['customer_request','policy_exception', 'tool_failures', 'other'],
                      'description': 'Why a human is needed'}}, ['reason']),
]

### check p22 & p05:

In [9]:
sc_fee_table(None); print("p22 fee table ok")
sc_reply_table(None); print("p05 reply table ok")

sc_schemas(SimpleNamespace(model=SimpleNamespace(tools_seen=TOOL_SCHEMAS)))
print("p21 schemas ok ")

p22 fee table ok
p05 reply table ok
p21 schemas ok 


### SalonAssistant:

In [10]:
MAX_MODEL_CALLS, MAX_TOOL_CALLS, MAX_MESSAGES = 6, 8, 24

_EMAIL_RE = re.compile(r'[\w.+-]+@[\w-]+(?:\.[\w-]+)+')
_DATE_OR_PHONE_RE = re.compile(
    r'(?P<date>\d{4}-\d{2}-\d{2}(?: \d{2}:\d{2})?)'
    r'|(?P<phone>\+?\(?\d{1,4}\)?[\s.-]?\d{2,4}[\s.-]?\d{4,})'
)

def _redact(text):
    """Hide e-mails and phone numbers (but keep ISO dates/times readable)."""
    text = _EMAIL_RE.sub('[email]', str(text))
    def repl(m):
        if m.group('date'):
            return m.group(0)
        return '[phone]' if sum(c.isdigit() for c in m.group(0)) >= 7 else m.group(0)
    return _DATE_OR_PHONE_RE.sub(repl, text)

def _euro(cents):
    return f'€{cents // 100}.{cents % 100:02d}'

_SCHEMA_BY_NAME = {t['function']['name']: t['function']['parameters'] for t in TOOL_SCHEMAS}

SLOT_MIN = 30
def parse_dt(s): return dt.datetime.strptime(s, '%Y-%m-%d %H:%M') 

def _validate_args(name, args):
    """Return an error message, or None when the arguments fit the tool's schema."""
    schema = _SCHEMA_BY_NAME[name]
    props = schema['properties']
    extra = set(args) - set(props)
    if extra:
        return f'unexpected arguments: {sorted(extra)}'
    for key in schema['required']:
        if key not in args:
            return f'missing argument: {key}'
    for key, value in args.items():
        spec = props[key]
        if not isinstance(value, str):
            return f'{key} must be a string'
        if 'enum' in spec and value not in spec['enum']:
            return f'{key} must be one of {spec["enum"]}'
        if 'pattern' in spec and not re.match(spec['pattern'], value):
            return f'{key} has the wrong format'
    try:
        if 'date' in args:
            dt.datetime.strptime(args['date'], '%Y-%m-%d')
        for key in ('start', 'new_start'):
            if key in args and parse_dt(args[key]).minute % SLOT_MIN:
                return f'{key} must be on a 30-minute mark'
    except ValueError:
        return 'the date/time is not a real calendar value'
    return None

def _result(status, error_type=None, data=None, attempts=1):
    return {'status': status, 'error_type': error_type, 'data': data, 'attempts': attempts}

In [17]:
class SalonAssistant:
    def __init__(self, model, backend, clock):
        self.model, self.backend, self.clock = model, backend, clock
        self.sessions, self.audit = {}, []

    def open_session(self, session_id, customer_id):
        self.sessions[session_id] = {
            'id': session_id, 'customer': customer_id,
            'messages': [],
            'pending': None,
            'turn': 0, 'seq': 0, 'seen_ids': set(),
            'error_count': 0,
            'evidence': set(),
            'write_success': False,
            'ticket_id': None,
            'last_user':[],
        }
    
    def handle(self, session_id, user_text):
        """Return the reply text for the customer."""
        st = self.sessions[session_id]
        st['turn'] += 1
        turn = {'model_calls': 0, 'tool_calls': 0}
        st['write_success'] = False
        st['error_count']=0
        st['last_user']= user_text
        st['ticket_id']= None
        user_text = user_text if isinstance(user_text, str) else str(user_text)
        self._audit(st, None, '(user)', None, None, 0, user_text)

        pending, st['pending'] = st['pending'], None
        st['messages'].append({'role': 'user', 'content': user_text})
        self._extract_evidence(st, user_text)
        if pending is not None and interpret_reply(user_text) == 'yes':
            self._execute_pending(st, turn, pending)


        reply = self._run_model(st, turn)
        if st['error_count'] >= 2:
            try:
                self._open_ticket(st, 'tool_failures',
                                  f"Repeated tool failures in session {st['id']}")
            except Exception:
                pass
        return reply
    
    def _system_prompt(self):
        now = self.clock()
        return (
            "You are Luna, the booking assistant of Studio Linden (a hair salon).\n"
            f"Now: {fmt(now)} ({now.strftime('%A')}). Times are local salon time.\n"
            "Rules:\n"
            "- Use the tools for every fact: services, prices, free slots and the customer's bookings.\n"
            "- book_appointment, reschedule_appointment and cancel_appointment only PROPOSE a change. "
            "After a proposal, ask the customer to confirm; nothing happens until they say yes in their next message.\n"
            "- Never say a booking was made, moved or cancelled unless a tool result says so.\n"
            "- Never state a time, price or booking id that did not come from a tool result or from the customer.\n"
            "- You only know the logged-in customer; never discuss anyone else's bookings.\n"
            "- If the customer wants a person, use handoff_to_human."
        )

    def _build_messages(self, st):
        msgs = st['messages']
        starts = [i for i, m in enumerate(msgs) if m['role'] == 'user']
        keep_from = starts[-1] if starts else 0
        for s in reversed(starts[:-1]):
            if len(msgs) - s <= MAX_MESSAGES - 1:
                keep_from = s
            else:
                break
        return [{'role': 'system', 'content': self._system_prompt()}] + msgs[keep_from:]

    def _finish(self, st, text):
        st['messages'].append({'role': 'assistant', 'content': text})
        return text

    def _run_model(self, st, turn):
        while True:
            if turn['model_calls'] >= MAX_MODEL_CALLS:
                return self._finish(st, SAFE_FALLBACK)
            turn['model_calls'] += 1
            try:
                resp = self.model(self._build_messages(st), TOOL_SCHEMAS)
            except Exception:
                return self._finish(st, SAFE_FALLBACK)

            if isinstance(resp, dict) and resp.get('tool_calls') is not None:
                tool_calls = resp['tool_calls']
                if not self._valid_batch(st, turn, tool_calls) or turn['model_calls'] >= MAX_MODEL_CALLS:
                    return self._finish(st, SAFE_FALLBACK)
                self._run_tool_calls(st, turn, tool_calls)
                continue
            if isinstance(resp, dict) and isinstance(resp.get('final'), str):
                text = resp['final']
                if not self._check_grounding(st , text):
                    return self._finish(st,SAFE_FALLBACK)
                return self._finish(st, text)
            return self._finish(st, SAFE_FALLBACK)

    def _valid_batch(self, st, turn, tool_calls):
        """The whole batch is rejected (nothing runs) if its structure is wrong."""
        if not isinstance(tool_calls, list) or not tool_calls:
            return False
        if turn['tool_calls'] + len(tool_calls) > MAX_TOOL_CALLS:
            return False
        ids = []
        for tc in tool_calls:
            if not isinstance(tc, dict):
                return False
            if not isinstance(tc.get('id'), str) or not tc['id'] or not isinstance(tc.get('name'), str):
                return False
            if not isinstance(tc.get('arguments'), (dict, str)):
                return False
            ids.append(tc['id'])
        return len(set(ids)) == len(ids) and not (set(ids) & st['seen_ids'])

    def _run_tool_calls(self, st, turn, tool_calls):
        calls = []
        for tc in tool_calls:
            args = tc['arguments'] if isinstance(tc['arguments'], str) else json.dumps(tc['arguments'])
            calls.append({'id': tc['id'], 'type': 'function', 'function': {'name': tc['name'], 'arguments': args}})
            st['seen_ids'].add(tc['id'])
        st['messages'].append({'role': 'assistant', 'content': None, 'tool_calls': calls})
        for tc in tool_calls:
            turn['tool_calls'] += 1
            res = self._run_one(st, tc['name'], tc['arguments'])
            self._record(st, tc['id'], tc['name'], res)

    def _record(self, st, call_id, tool, res):
        """Show the result to the model and write the audit event."""
        obs = {'call_id': call_id, 'tool': tool, 'status': res['status']}
        if res['error_type']:
            obs['error_type'] = res['error_type']
        obs['data'] = res['data']
        st['messages'].append({'role': 'tool', 'tool_call_id': call_id,
                               'content': json.dumps(obs, ensure_ascii=False)})
        preview = '' if res['data'] is None else json.dumps(res['data'], ensure_ascii=False)
        self._audit(st, call_id, tool, res['status'], res['error_type'], res['attempts'], preview)
        if res['status'] == 'error' and res['error_type'] in ('timeout', 'unknown_outcome'):
            st['error_count'] = st.get('error_count', 0) + 1
        if res['data'] is not None:
            self._extract_evidence(st, json.dumps(res['data'], ensure_ascii=False))

    def _audit(self, st, call_id, tool, status, error_type, attempts, preview):
        self.audit.append({'session': st['id'], 'turn': st['turn'], 'call_id': call_id, 'tool': tool,
                           'status': status, 'error_type': error_type, 'attempts': attempts,
                           'preview': _redact(preview)[:80]})

 
    def _run_one(self, st, name, arguments):
        if name not in _SCHEMA_BY_NAME:
            return _result('error', 'unknown_tool', attempts=0)
        args = arguments
        if isinstance(args, str):
            try:
                args = json.loads(args)
            except (ValueError, TypeError):
                return _result('error', 'parse_error', attempts=0)
        if not isinstance(args, dict):
            return _result('error', 'validation_error', {'detail': 'arguments must be an object'}, 0)
        problem = _validate_args(name, args)
        if problem:
            return _result('error', 'validation_error', {'detail': problem}, 0)
        handler = {
            'list_services': self._t_list_services,
            'find_slots': self._t_find_slots,
            'get_my_bookings': self._t_get_my_bookings,
            'book_appointment': self._t_book,
            'reschedule_appointment': self._t_reschedule,
            'cancel_appointment': self._t_cancel,
            'handoff_to_human': self._t_handoff,
        }.get(name)
        if handler is None:
            return _result('error', 'not_implemented', attempts=0)
        try:
            return handler(st, args)
        except BackendTimeout:
            if name in ('list_services', 'find_slots', 'get_my_bookings'):
                try:
                    res = handler(st, args)
                    res['attempts'] = 2       
                    return res
                except BackendTimeout:
                    return _result('error', 'timeout', attempts=2)
            return _result('error', 'timeout', attempts=1)            
                
        except SlotTaken:
            return _result('error', 'slot_taken', attempts=1)
        except BackendError as exc:
            return _result('error', exc.code, attempts=1)
        except Exception as exc:
            return _result('error', 'unexpected_error', {'detail': type(exc).__name__})
        
    def _t_list_services(self, st, args):
        services = self.backend.list_services()
        data = [{'service_id': sid, 'name': s['name'], 'minutes': s['minutes'],
                 'price_cents': s['price_cents'], 'price': _euro(s['price_cents'])}
                for sid, s in services.items()]
        return _result('ok', data=data)

    def _t_find_slots(self, st, args):
        slots = self.backend.free_slots(args['service_id'], args['date'], args.get('stylist_id'))
        return _result('ok' if slots else 'not_found', data=slots)

    def _t_get_my_bookings(self, st, args):
        rows = self.backend.bookings_for(st['customer'])
        data = [{k: b[k] for k in ('booking_id', 'service_id', 'stylist_id', 'start', 'status', 'reschedules')}
                for b in rows]
        return _result('ok', data=data)

   
    def _alternatives(self, service_id, date, wanted_start, stylist_id=None, limit=3):
        slots = self.backend.free_slots(service_id, date, stylist_id)
        want = parse_dt(wanted_start)
        slots.sort(key=lambda r: (abs((parse_dt(r['start']) - want).total_seconds()),
                                  r['start'], r['stylist_id']))
        return slots[:limit]

    def _own_booking(self, st, booking_id):
        mine = self.backend.bookings_for(st['customer'])
        return next((b for b in mine if b['booking_id'] == booking_id), None), mine

    def _t_book(self, st, args):
        if st['pending'] is not None:
            return _result('blocked', 'pending_exists', {'detail': 'another change is already waiting for confirmation'}, 0)
        service_id, stylist_id, start = args['service_id'], args['stylist_id'], args['start']
        service, stylist = SERVICES[service_id], STYLISTS[stylist_id]
        if service_id not in stylist['services']:
            return _result('blocked', 'stylist_not_offering',
                           {'detail': f"{stylist['name']} does not offer {service['name']}"}, 0)

        mine = [b for b in self.backend.bookings_for(st['customer']) if b['status'] == 'active']
        if len(mine) >= MAX_ACTIVE:
            return _result('blocked', 'limit_reached',
                           {'detail': f'at most {MAX_ACTIVE} active bookings are allowed'}, 0)
        new_s = parse_dt(start)
        new_e = new_s + dt.timedelta(minutes=service['minutes'])
        for b in mine:
            s = parse_dt(b['start'])
            e = s + dt.timedelta(minutes=SERVICES[b['service_id']]['minutes'])
            if new_s < e and s < new_e:
                return _result('blocked', 'overlap',
                               {'booking_id': b['booking_id'],
                                'detail': 'overlaps another booking of the customer'}, 0)

        date = start[:10]
        free = self.backend.free_slots(service_id, date)
        if {'stylist_id': stylist_id, 'start': start} not in free:
            alts = self._alternatives(service_id, date, start)
            return _result('blocked', 'slot_unavailable', {'alternatives': alts})

        summary = (f"Book {service['name']} with {stylist['name']} on {date} at {start[11:]} "
                   f"for {_euro(service['price_cents'])}.")
        st['pending'] = {'tool': 'book_appointment', 'args': dict(args), 'summary': summary}
        return _result('needs_confirmation', data={'summary': summary})

   
    def _t_reschedule(self, st, args):
        if st['pending'] is not None:
            return _result('blocked', 'pending_exists', {'detail': 'another change is already waiting for confirmation'}, 0)
        bid, new_start = args['booking_id'], args['new_start']
        b, mine = self._own_booking(st, bid)
        if b is None:
            return _result('not_found', attempts=0)
        if b['status'] != 'active':
            return _result('error', 'already_cancelled', {'booking_id': bid}, 0)

        old_s = parse_dt(b['start'])
        until = old_s - self.clock()
        if until <= dt.timedelta(0):
            return _result('blocked', 'already_started', {'booking_id': bid}, 0)
        if until < dt.timedelta(hours=24):
            return _result('blocked', 'too_late', {'booking_id': bid}, 0)
        if b['reschedules'] >= 1:
            return _result('blocked', 'already_rescheduled', {'booking_id': bid}, 0)
        if new_start == b['start']:
            return _result('blocked', 'same_slot', {'booking_id': bid}, 0)

        service, stylist = SERVICES[b['service_id']], STYLISTS[b['stylist_id']]
        old_e = old_s + dt.timedelta(minutes=service['minutes'])
        new_s = parse_dt(new_start)
        new_e = new_s + dt.timedelta(minutes=service['minutes'])
        for other in mine:
            if other['booking_id'] == bid or other['status'] != 'active':
                continue
            s = parse_dt(other['start'])
            e = s + dt.timedelta(minutes=SERVICES[other['service_id']]['minutes'])
            if new_s < e and s < new_e:
                return _result('blocked', 'overlap',
                               {'booking_id': other['booking_id']}, 0)

        date = new_start[:10]
        free = self.backend.free_slots(b['service_id'], date, b['stylist_id'])
        overlaps_itself = new_s < old_e and old_s < new_e
        if {'stylist_id': b['stylist_id'], 'start': new_start} not in free and not overlaps_itself:
            alts = self._alternatives(b['service_id'], date, new_start, b['stylist_id'])
            return _result('blocked', 'slot_unavailable',
                           {'booking_id': bid, 'alternatives': alts})

        summary = (f"Move {bid} ({service['name']} with {stylist['name']}) "
                   f"from {b['start'][:10]} at {b['start'][11:]} to {date} at {new_start[11:]}.")
        st['pending'] = {'tool': 'reschedule_appointment', 'args': dict(args), 'summary': summary}
        return _result('needs_confirmation', data={'summary': summary})

    def _t_cancel(self, st, args):
        if st['pending'] is not None:
            return _result('blocked', 'pending_exists', {'detail': 'another change is already waiting for confirmation'}, 0)
        bid = args['booking_id']
        b, _ = self._own_booking(st, bid)
        if b is None:
            return _result('not_found', attempts=0)
        if b['status'] != 'active':
            return _result('error', 'already_cancelled', {'booking_id': bid}, 0)

        service, stylist = SERVICES[b['service_id']], STYLISTS[b['stylist_id']]
        try:
            fee = cancellation_fee_cents(service['price_cents'], parse_dt(b['start']), self.clock())
        except ValueError:
            return _result('blocked', 'already_started', {'booking_id': bid}, 0)

        what = (f"Cancel {bid} ({service['name']} with {stylist['name']} "
                f"on {b['start'][:10]} at {b['start'][11:]}).")
        summary = what + (' Cancelling is free of charge.' if fee == 0
                          else f' Cancellation fee: {_euro(fee)}.')
        st['pending'] = {'tool': 'cancel_appointment', 'args': dict(args),
                         'summary': summary, 'fee_cents': fee}
        return _result('needs_confirmation', data={'summary': summary, 'fee_cents': fee})

    def _open_ticket(self , st , reason , summary):
        """at most one ticket per customer message(R11); later calls reuse  the first ticket."""
        if st['ticket_id'] is None:
            st['ticket_id']=self.backend.create_ticket({
                'customer_id': st['customer'],
                'reason':reason,
                'summary':_redact(summary)[:300],
            })
        return st['ticket_id']
    
    def _t_handoff(self, st, args):
        reason = args.get('reason', 'other')
        summary = f"Customer {st['customer']} asked for a human ({reason}). Last message: {st['last_user']}"
        try:
            return _result('ok', data={'ticket_id': self._open_ticket(st, reason, summary)})
        except Exception as exc:
            return _result('error', 'unexpected_error', {'detail': type(exc).__name__})


    _BLOCKING_CODES = {'limit_reached', 'too_late', 'already_rescheduled',
                       'already_started', 'overlap', 'same_slot',
                       'stylist_not_offering', 'slot_unavailable', 'pending_exists'}

    def _execute_pending(self, st, turn, pending):
        """The customer said yes: run the stored action ourselves and show the model a synthetic tool call."""
        st['seq'] += 1
        call_id = f"exec-{st['turn']}-{st['seq']}"
        while call_id in st['seen_ids']:
            call_id += 'x'
        st['seen_ids'].add(call_id)
        turn['tool_calls'] += 1
        name, args = pending['tool'], pending['args']
        st['messages'].append({'role': 'assistant', 'content': None, 'tool_calls': [
            {'id': call_id, 'type': 'function',
             'function': {'name': name, 'arguments': json.dumps(args)}}]})
        request_key = f"{st['id']}:{st['turn']}:{st['seq']}"
        fields = ('booking_id', 'service_id', 'stylist_id', 'start', 'status')
        
        def _do_write():
            if name == 'book_appointment':
                done = self.backend.book(st['customer'], args['service_id'],
                                         args['stylist_id'], args['start'], request_key)
                return _result('ok', data={k: done[k] for k in fields})
            if name == 'reschedule_appointment':
                done = self.backend.reschedule(st['customer'], args['booking_id'],
                                               args['new_start'], request_key)
                return _result('ok', data={k: done[k] for k in fields})
            done = self.backend.cancel(st['customer'], args['booking_id'],
                                       pending['fee_cents'], request_key)
            return _result('ok', data=done)

        try:
            res = _do_write()
        except SlotTaken:
            alts = self._alternatives(
                args.get('service_id'),
                args.get('start', args.get('new_start', ''))[:10],
                args.get('start', args.get('new_start', '')),
                
            )
            res = _result('error', 'slot_taken', {'alternatives': alts})
        except BackendTimeout:
            found = self.backend.find_by_key(request_key)
            if found:
                data = {k: found[k] for k in fields if k in found}
                res = _result('ok', data=data, attempts=2)
            else:
                try:
                    res = _do_write()
                    res['attempts'] = 2
                except BackendTimeout:
                    res = _result('error', 'unknown_outcome', attempts=2)
                except (SlotTaken, BackendError) as exc:
                    code = exc.code if hasattr(exc, 'code') else 'slot_taken'
                    kind = 'blocked' if code in self._BLOCKING_CODES else 'error'
                    res = _result(kind, code, {'detail': 'second attempt failed'}, 2)
        except BackendError as exc:
            kind = 'blocked' if exc.code in self._BLOCKING_CODES else 'error'
            res = _result(kind, exc.code, {'detail': 'the salon system refused the change'})
        except Exception as exc:
            res = _result('error', 'unexpected_error', {'detail': type(exc).__name__})

        self._record(st, call_id, name, res)
        if res['data'] is not None:
            self._extract_evidence(st, json.dumps(res['data'], ensure_ascii=False))
            if res['status'] == 'ok':    
                st['write_success'] = True

    def _extract_evidence(self, st, text):
            for m in re.finditer(r'\b\d{1,2}:\d{2}\b', text):
                st['evidence'].add(m.group(0))
            for m in re.finditer(r'€\d+(?:\.\d{2})?', text):
                st['evidence'].add(m.group(0))
            for m in re.finditer(r'\bB-\d+\b', text):
                st['evidence'].add(m.group(0))

    def _check_grounding(self, st, text):
            for m in re.finditer(r'\b\d{1,2}:\d{2}\b', text):
                if m.group(0) not in st['evidence']:
                    return False
            for m in re.finditer(r'€\d+(?:\.\d{2})?', text):
                if m.group(0) not in st['evidence']:
                    return False
            for m in re.finditer(r'\bB-\d+\b', text):
                if m.group(0) not in st['evidence']:
                    return False
            if re.search(r'\b(booked|rescheduled|cancelled|moved|confirmed)\b', text, re.I):
                if not st.get('write_success'):
                    return False
            return True   

### checking :

In [12]:
for sid in ('P01','P02','P03'):
    trace(sid , SalonAssistant)

=== P01 Greeting needs no tools ===

[turn 1 ('Hi!')]
  USER : Hi!
  REPLY: Hello! How can I help you today?

RESULT: PASS
=== P02 Browse: two independent reads in one turn ===

[turn 1 ("What do you offer, and what'")]
  USER : What do you offer, and what's free for a haircut on Tuesday?
  REPLY: We offer Haircut (€35.00), Beard trim, Wash & style, Colour and Highlights. Tuesday haircut slots start at 09:00 and 09:30.

RESULT: PASS
=== P03 Book: propose, then confirm in a LATER turn ===

[turn 1 ('Please book me a haircut wit')]
  USER : Please book me a haircut with Mara tomorrow at 11:00
  REPLY: Here is what I will book: Haircut with Mara on 2026-10-20 at 11:00 for €35.00. Shall I go ahead?

[turn 2 ('yes please')]
  USER : yes please
  REPLY: Done! Your haircut with Mara is booked for 2026-10-20 at 11:00 (B-7).

RESULT: PASS


In [13]:
satge3= [s for s in PUBLIC if s['id'] in {'P07','P08','P09','P10','P11','P19'}]
run_suite(satge3 , SalonAssistant , title='stage 3')
trace('P08' , SalonAssistant)

✓ P07 Cancel more than 24h ahead is free 
✓ P08 Late cancellation: fee shown BEFORE the customer says yes 
✓ P09 Reschedule: allowed at exactly 24h, only once 
✓ P10 Reschedule inside 24h is refused 
✓ P11 Identity: another customer's booking 
✓ P19 Two requests in one message: one pending action 
stage 3: 6/6 scenarios passed
=== P08 Late cancellation: fee shown BEFORE the customer says yes ===

[turn 1 ('Cancel my appointment today')]
  USER : Cancel my appointment today
  REPLY: Cancelling B-3 within 24 hours costs €10.00. Do you want to proceed?

[turn 2 ('yes')]
  USER : yes
  REPLY: B-3 is cancelled. The €10.00 fee applies.

RESULT: PASS


In [14]:
run_public(SalonAssistant)

✓ P01 Greeting needs no tools 
✓ P02 Browse: two independent reads in one turn 
✓ P03 Book: propose, then confirm in a LATER turn 
✓ P04 "yes, but..." and a bare "yes" never book 
✓ P05 interpret_reply: 36 customer replies 
✓ P06 Race: the slot is taken between proposal and "yes" 
✓ P07 Cancel more than 24h ahead is free 
✓ P08 Late cancellation: fee shown BEFORE the customer says yes 
✓ P09 Reschedule: allowed at exactly 24h, only once 
✓ P10 Reschedule inside 24h is refused 
✓ P11 Identity: another customer's booking 
✓ P12 Booking limit is checked before asking 
✓ P13 Lost response after a committed booking 
✓ P14 A read timeout is retried once 
✓ P15 Repeated failures: honest reply and human handoff 
✓ P16 Invented prices, invented success, and a true answer 
✓ P17 Malformed, unknown and smuggling tool calls 
✓ P18 30 chatty turns, then a booking (context window) 
✓ P19 Two requests in one message: one pending action 
✓ P20 Human handoff without leaking personal data 
✓ P21 Tool sc

39.99999999999999

### my scenarios:

In [15]:
def _book_obs(ctx):
    return [o for o in ctx.model.observations() if o.get('tool') == 'book_appointment']

def _only_one_booking(ctx):
    assert ctx.backend.write_log == [('book', 'B-7')], ctx.backend.write_log

def chk_write_fails_twice(ctx):
    last = _book_obs(ctx)[-1]
    assert last['status'] == 'error' and last['error_type'] == 'unknown_outcome', last
    assert ctx.backend.write_log == [] and ctx.backend.call_log.count('book') == 2, ctx.backend.call_log
    assert ctx.replies[1] == SAFE_FALLBACK, 'a reply that claims success without proof must be replaced'

def chk_retry_same_key(ctx):
    _only_one_booking(ctx)
    assert ctx.backend.call_log.count('book') == 2 and _book_obs(ctx)[-1]['status'] == 'ok'

def chk_applied_but_unknown(ctx):
    assert len(new_bookings(ctx.backend)) == 1, 'the retry used the same key, so there is still exactly one booking'
    assert _book_obs(ctx)[-1]['error_type'] == 'unknown_outcome'

def chk_fee_changes(ctx):
    assert ctx.backend.bookings['B-3']['status'] == 'active' and ctx.backend.write_log == []
    last = [o for o in ctx.model.observations() if o.get('tool') == 'cancel_appointment'][-1]
    assert last['status'] == 'error' and last['error_type'] == 'fee_mismatch', last

def chk_repeat_proposal(ctx):
    _only_one_booking(ctx)
    last = _book_obs(ctx)[-1]
    assert last['call_id'] == 'again' and last['status'] == 'blocked', last

def chk_change_of_mind(ctx):
    b = new_bookings(ctx.backend)
    assert [(x['start'], x['stylist_id']) for x in b] == [('2026-10-21 14:00', 'mara')], b

def chk_invented_time(ctx):
    assert ctx.replies[0].startswith('Tuesday haircut slots start at 09:00'), ctx.replies[0]
    assert ctx.replies[1] == SAFE_FALLBACK, ctx.replies[1]

def chk_one_ticket(ctx):
    assert len(ctx.backend.tickets) == 1, ctx.backend.tickets
    t = ctx.backend.tickets[0]
    assert len(t['summary']) <= 300 and '@' not in t['summary'] and '5551234' not in t['summary'], t

def chk_old_id_reused(ctx):
    assert ctx.replies[1] == SAFE_FALLBACK and 'bookings_for' not in ctx.backend.call_log, ctx.backend.call_log

def _six_model_calls(ctx):
    assert ctx.model.n_calls == 6, f'{ctx.model.n_calls} model calls in one customer message (limit 6)'

def chk_limits(ctx):
    assert ctx.replies == [SAFE_FALLBACK, SAFE_FALLBACK], ctx.replies
    assert ctx.model.n_calls == 7, ctx.model.n_calls
    assert ctx.backend.call_log.count('list_services') <= 6, 'a batch of nine tool calls must not run at all'

def chk_audit(ctx):
    keys = {'session', 'turn', 'call_id', 'tool', 'status', 'error_type', 'attempts', 'preview'}
    assert all(set(e) == keys for e in ctx.assistant.audit), [set(e) for e in ctx.assistant.audit][:2]
    blob = json.dumps(ctx.assistant.audit)
    assert '@' not in blob and '5551234567' not in blob, 'e-mail / phone leaked into the audit log'
    slots = [e for e in ctx.assistant.audit if e['tool'] == 'find_slots']
    assert slots and '2026-10-20 09:00' in slots[0]['preview'], slots
    assert all(len(e['preview']) <= 80 for e in ctx.assistant.audit)

_two_fail = lambda b: b.faults.update(fail_next_write=2)

MY_SCENARIOS = [
 {'id': 'M01', 'title': 'Write fails twice: unknown_outcome, a success claim is blocked, no duplicate', 'weight': 1, 'turns': [_propose_cut(),T('yes', final('Done! Your haircut with Mara is booked for 2026-10-20 at 11:00 (B-7).'), before=_two_fail)], 'check': chk_write_fails_twice},
 {'id': 'M02', 'title': 'Write times out once: retried with the SAME key, exactly one booking', 'weight': 1, 'turns': [_propose_cut(),
      T('yes', final('All set: haircut with Mara on 2026-10-20 at 11:00, booking B-7.'), has=['B-7'],
        before=lambda b: b.faults.update(fail_next_write=1))], 'check': chk_retry_same_key},
 {'id': 'M03', 'title': 'Retry commits but its reply is lost: unknown_outcome, still ONE booking', 'weight': 1, 'turns': [_propose_cut(),
      T('yes', final("I couldn't confirm whether the booking was saved. I'll ask a team member to check."),
        before=lambda b: b.faults.update(fail_next_write=1, lose_next_write_response=1))], 'check': chk_applied_but_unknown},
 {'id': 'M04', 'title': 'The cancellation fee changes while the customer decides: nothing is cancelled', 'weight': 1, 'customer': 'c_ben', 'turns': [
      T('Cancel my appointment today', calls(call('c1', 'cancel_appointment', booking_id='B-3')),
        final('Cancelling B-3 costs €10.00. Do you want to proceed?'), has=['€10.00']),
      T('yes', final('The fee has changed, so I have not cancelled B-3. Shall I look at it again?'),
        before=lambda b: setattr(b, 'now', b.now + dt.timedelta(hours=4)))], 'check': chk_fee_changes},
 {'id': 'M05', 'title': 'After a confirmed booking, the model proposes the same booking again: blocked', 'weight': 1, 'turns': [_propose_cut(),
      T('yes', calls(call('again', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-20 11:00')),
        final('Your haircut is booked for 2026-10-20 at 11:00 (B-7).'), has=['B-7'])], 'check': chk_repeat_proposal},
 {'id': 'M06', 'title': 'Customer changes their mind: only the last confirmed proposal is booked', 'weight': 1, 'turns': [_propose_cut(),
      T('actually Wednesday at 14:00 please', calls(call('w1', 'book_appointment', service_id='cut', stylist_id='mara', start='2026-10-21 14:00')),
        final('Haircut with Mara on 2026-10-21 at 14:00 for €35.00. Shall I book it?'), has=['Shall I']),
      T('yes', final('Your haircut is booked for 2026-10-21 at 14:00 (B-7).'), has=['B-7'])], 'check': chk_change_of_mind},
 {'id': 'M07', 'title': 'A reply with a time the tools never returned is replaced by the safe fallback', 'weight': 1, 'turns': [
      T('Haircut slots on Tuesday?', calls(call('s1', 'find_slots', service_id='cut', date='2026-10-20')),
        final('Tuesday haircut slots start at 09:00 and 09:30.')),
      T('And the latest one?', final('The latest slot is at 16:45.'))], 'check': chk_invented_time},
 {'id': 'M08', 'title': 'Two handoff calls and a very long message: one ticket, short, no e-mail or phone', 'weight': 1, 'turns': [
      T('I want to talk to a person ' + 'please ' * 60 + 'mail me at bea@example.com or call +49 151 5551234',
        calls(call('h1', 'handoff_to_human', reason='customer_request'), call('h2', 'handoff_to_human', reason='customer_request')),
        final("I've passed your request to our team."))], 'check': chk_one_ticket},
 {'id': 'M09', 'title': 'A tool-call id reused from an earlier turn: the whole batch is rejected', 'weight': 1, 'turns': [
      T('What do you offer?', calls(call('a1', 'list_services')), final('We offer Haircut (€35.00) and more.')),
      T('And my bookings?', calls(call('a1', 'get_my_bookings')))], 'check': chk_old_id_reused},
 {'id': 'M10', 'title': 'A model that never stops calling tools hits the call limits', 'weight': 1, 'loose': True, 'turns': [
      T('Tell me everything', *[calls(call(f'n{i}', 'list_services')) for i in range(1, 8)], final('done'), after=_six_model_calls),
      T('Again', calls(*[call(f'k{i}', 'list_services') for i in range(1, 10)]))], 'check': chk_limits},
 {'id': 'M11', 'title': 'Audit log: exact keys, e-mail/phone hidden, dates kept, previews <= 80 characters', 'weight': 1, 'turns': [
      T('Hi, my email is bea@example.com and my phone is 555 1234567. Free cuts on Tuesday?',
        calls(call('f1', 'find_slots', service_id='cut', date='2026-10-20')), final('Tuesday slots start at 09:00.'))],
      'check': chk_audit},
]

In [18]:
run_suite(MY_SCENARIOS, SalonAssistant, title='stress tests')

✓ M01 Write fails twice: unknown_outcome, a success claim is blocked, no duplicate 
✓ M02 Write times out once: retried with the SAME key, exactly one booking 
✓ M03 Retry commits but its reply is lost: unknown_outcome, still ONE booking 
✓ M04 The cancellation fee changes while the customer decides: nothing is cancelled 
✓ M05 After a confirmed booking, the model proposes the same booking again: blocked 
✓ M06 Customer changes their mind: only the last confirmed proposal is booked 
✓ M07 A reply with a time the tools never returned is replaced by the safe fallback 
✓ M08 Two handoff calls and a very long message: one ticket, short, no e-mail or phone 
✓ M09 A tool-call id reused from an earlier turn: the whole batch is rejected 
✓ M10 A model that never stops calling tools hits the call limits 
✓ M11 Audit log: exact keys, e-mail/phone hidden, dates kept, previews <= 80 characters 
stress tests: 11/11 scenarios passed


11.0